# 4-5. Airgap 전환: `.env` 한 줄로 폐쇄망 모드

⏱ **소요시간**: 15분

## 학습 목표

- `common.get_llm()` / `common.get_embeddings()`의 provider 스위칭 구조를 이해한다.
- `.env` 파일의 `LLM_PROVIDER`, `EMBEDDING_PROVIDER`만 바꿔 **Day1 S2-4, Day2 S3-5 노트북 전체를 폐쇄망 모드로 재실행**하는 방법을 체득한다.
- 🔒 **vLLM**의 OpenAI 호환 서버 개념과 PagedAttention 개요를 이해한다.
- 마지막 `과정 완료 체크리스트`로 2일 커리큘럼을 마무리한다.

## 핵심 키워드

`.env` · `LLM_PROVIDER` · `EMBEDDING_PROVIDER` · `Ollama` · `HuggingFace Embeddings` · `vLLM` · `PagedAttention` · `OpenAI 호환 엔드포인트` · `커널 재시작`

## 배지

- 🔒 Ollama (로컬 LLM)
- 🔒 BGE-M3 / bge-m3 (로컬 임베딩, HuggingFace)
- 🔒 vLLM (고성능 추론 서버, OpenAI 호환)

In [ ]:
import sys
sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge

# 현재 모드 확인
print("[현재 상태]", provider_badge())

## 1. `.env` 수정법

저장소 루트의 `.env` 파일을 열어 아래 두 줄만 바꾸면 된다.

### Before (☁️ Cloud)

```ini
LLM_PROVIDER=openai
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini

EMBEDDING_PROVIDER=openai
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

### After (🔒 Airgap)

```ini
LLM_PROVIDER=ollama
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:7b-instruct

EMBEDDING_PROVIDER=local
LOCAL_EMBEDDING_MODEL=BAAI/bge-m3
```

> 변경 후 **주피터 커널을 재시작**해야 `load_dotenv()`가 새 값으로 다시 읽힌다.
> (또는 `import importlib, common; importlib.reload(common.llm)` + `lru_cache` clear)

In [ ]:
# 변경 전/후 비교용: 환경변수를 런타임에 덮어써 시뮬레이션
import os, importlib
from common import llm as llm_module

def snapshot():
    return {
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "(unset)"),
        "EMBEDDING_PROVIDER": os.getenv("EMBEDDING_PROVIDER", "(unset)"),
        "OPENAI_MODEL": os.getenv("OPENAI_MODEL", "(unset)"),
        "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL", "(unset)"),
        "badge": llm_module.provider_badge(),
    }

print("[BEFORE]")
for k, v in snapshot().items():
    print(f"  {k} = {v}")

# --- 전환 시뮬레이션: .env를 수정했다고 가정하고 값 덮어쓰기 ---
os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "qwen2.5:7b-instruct"
os.environ["EMBEDDING_PROVIDER"] = "local"
os.environ["LOCAL_EMBEDDING_MODEL"] = "BAAI/bge-m3"
llm_module.get_embeddings.cache_clear()  # lru_cache 비우기

print("\n[AFTER]")
for k, v in snapshot().items():
    print(f"  {k} = {v}")

## 2. 재실행 가이드: Day1 S2-4, Day2 S3-5

`.env` 수정 후 아래 노트북들을 **커널 재시작 → Run All**만 하면 동일한 파이프라인이 로컬 모델로 재현된다.

| 노트북 | 경로 | 확인 포인트 |
| --- | --- | --- |
| Day1 S2 RAG 기초 | `day1/session2_rag_basics/` | 벡터스토어 재빌드 필요 (임베딩 차원 변경 시) |
| Day1 S3 프롬프트 엔지니어링 | `day1/session3_prompting/` | 출력 톤/형식 차이 비교 |
| Day1 S4 평가 | `day1/session4_eval/` | recall@k, MRR 지표 차이 |
| Day2 S1 도메인 튜닝 | `day2/session1_*` | fine-tuning 결과 재사용 |
| Day2 S2 에이전트 | `day2/session2_*` | 툴 호출 파싱 형식 주의 (모델별) |
| Day2 S3 Advanced RAG / GraphRAG | `day2/session3_advanced_graphrag/` | Neo4j 인덱스 재생성 |
| Day2 S4 Security + Web | `day2/session4_security_webservice/` | Presidio/NeMo는 provider 무관 |

### 주의 사항

1. **임베딩 차원 불일치** — OpenAI `text-embedding-3-small`(1536d) ↔ `BAAI/bge-m3`(1024d). 벡터스토어 인덱스를 **새로 빌드**해야 한다.
2. **토큰/성능 차이** — 로컬 7B 모델은 frontier 모델 대비 추론 품질·JSON 준수·긴 컨텍스트 안정성이 떨어질 수 있다. 프롬프트를 더 명시적으로 작성.
3. **툴 호출 포맷** — Ollama 모델은 `tools` 포맷 지원 여부가 모델마다 다르다. LangChain `bind_tools`로 감싸기.
4. **컨텍스트 길이** — `num_ctx` (Ollama), `--max-model-len` (vLLM) 명시적 설정.
5. **첫 호출 지연** — 모델 로딩 때문에 첫 호출은 10~60초 걸릴 수 있다 (`ollama run`으로 웜업 권장).

In [ ]:
# 작은 예제로 전/후 비교 — 주의: 아래 코드는 해당 provider가 실제 동작하는 환경에서만 응답이 나옵니다.
from langchain_core.messages import HumanMessage

QUESTION = "청약철회 기간은?"


def try_invoke(label: str, **env_overrides):
    prev = {k: os.environ.get(k) for k in env_overrides}
    os.environ.update({k: v for k, v in env_overrides.items() if v is not None})
    llm_module.get_embeddings.cache_clear()
    try:
        llm = llm_module.get_llm(temperature=0.0)
        resp = llm.invoke([HumanMessage(content=QUESTION)])
        content = resp.content if isinstance(resp.content, str) else str(resp.content)
        print(f"\n[{label}] badge={llm_module.provider_badge()}")
        print(content[:300])
    except Exception as e:
        print(f"\n[{label}] 실행 실패: {type(e).__name__}: {str(e)[:200]}")
    finally:
        # 환경 복원
        for k, v in prev.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v


# Before (Cloud)
try_invoke("BEFORE: ☁️ Cloud", LLM_PROVIDER="openai", OPENAI_MODEL=os.getenv("OPENAI_MODEL", "gpt-4o-mini"))

# After (Airgap)
try_invoke("AFTER:  🔒 Airgap", LLM_PROVIDER="ollama", OLLAMA_MODEL="qwen2.5:7b-instruct")

## 3. vLLM 개념 소개 🔒

**vLLM** (UC Berkeley Sky Lab)은 고처리량 LLM 추론 서버이다. 기관 규모 서빙에 Ollama보다 적합한 경우가 많다.

### 핵심 특징

1. **PagedAttention** — KV 캐시를 OS 가상메모리처럼 **고정 크기 페이지**로 관리. 메모리 단편화 해소, 동일 프롬프트 재사용 공유.
2. **Continuous Batching** — 요청별 길이가 달라도 토큰 단계에서 동적으로 배칭 → 처리량 극대화.
3. **OpenAI 호환 서버** — `POST /v1/chat/completions`, `POST /v1/completions`, `POST /v1/embeddings` 기본 제공.
4. **양자화 지원** — AWQ, GPTQ, FP8, BitsAndBytes.
5. **멀티 GPU** — Tensor/Pipeline parallelism, Speculative decoding.

### 기동 예시 (외부망에서 모델을 반입했다고 가정)

```bash
# vLLM OpenAI 호환 서버
vllm serve /models/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 --port 8000 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.9 \
    --served-model-name qwen2.5-7b
```

### 기존 코드에서 사용하기

`common.get_llm()`에 vLLM 분기를 추가하거나, 가장 간단하게는 OpenAI 클라이언트의 `base_url`만 바꾼다.

```ini
# .env
LLM_PROVIDER=openai            # ← OpenAI SDK 재사용
OPENAI_API_KEY=dummy-not-used
OPENAI_BASE_URL=http://vllm.internal:8000/v1
OPENAI_MODEL=qwen2.5-7b
```

### Ollama vs vLLM 가이드

| 상황 | 추천 |
| --- | --- |
| 1~수십 명 사내 PoC, CPU+GPU 혼용 | **Ollama** (간편, GGUF 양자화 풍부) |
| 수백 동시 요청, 고성능 GPU 서버 | **vLLM** (continuous batching) |
| 개발자 개인 노트북 | **Ollama** |
| 프로덕션 API 백엔드 | **vLLM** + K8s |
| 모델 실험/빠른 전환 | **Ollama** (`ollama pull`) |

## 4. 과정 완료 체크리스트 ✅

다음 10개 항목을 모두 스스로 수행·설명할 수 있으면 본 2일 과정을 완주한 것이다.

- [ ] **1. 규제 매핑** — 개인정보보호법·신용정보법·전자금융거래법의 핵심 조항을 LLM 서비스 설계 체크리스트로 전환할 수 있다.
- [ ] **2. 망분리 아키텍처** — 인터넷망 / DMZ / 내부망 구조 위에 LLM 추론·벡터DB·감사로그 배치를 그림으로 설명할 수 있다.
- [ ] **3. PII 마스킹** — Presidio + 한국어 커스텀 PatternRecognizer로 RRN/BRN/ACCOUNT를 탐지·마스킹할 수 있다.
- [ ] **4. 가드레일** — NeMo Guardrails로 입·출력 rail을 구성하고 OWASP LLM Top10의 주요 위협에 매핑할 수 있다.
- [ ] **5. 감사 로그** — trace_id / user / query / context_sources / pii_flags 스키마로 JSONL 감사 로그를 기록·조회할 수 있다.
- [ ] **6. OpenWebUI + Ollama 운영** — Docker Compose 기동, 모델 pull, 지식베이스 생성, Pipelines 연동을 수행할 수 있다.
- [ ] **7. 폐쇄망 반입 절차** — 모델 tar.gz + sha256 + 자료전송시스템 경유 + 내부 로드 흐름을 직접 재현할 수 있다.
- [ ] **8. 오프라인 의존성** — `pip download` + `--no-index --find-links`로 파이썬 패키지를 오프라인 설치할 수 있다.
- [ ] **9. 커스텀 서비스** — FastAPI로 `/chat` SSE 및 OpenAI 호환 `/v1/chat/completions`를 구현하고 Gradio UI를 연결할 수 있다.
- [ ] **10. Airgap 전환** — `.env` 한 줄(`LLM_PROVIDER`/`EMBEDDING_PROVIDER`) 변경으로 Day1·Day2 노트북을 로컬 모델로 재실행할 수 있다.
